# Topological Sort

## Pattern Overview

**Topological Sort** produces a **linear ordering of vertices** in a Directed Acyclic Graph (DAG)
such that for every directed edge u → v, vertex u appears before vertex v in the ordering.

### Key Insight
- Only valid on **DAGs** (Directed Acyclic Graphs)
- Multiple valid orderings may exist
- **Cycle detection**: if result length < n, a cycle exists

---

## Algorithm 1: Kahn's Algorithm (BFS / Indegree)

1. Compute indegree for every node
2. Enqueue all nodes with indegree = 0
3. Process queue: pop node, append to result, decrement neighbors' indegrees
4. If a neighbor's indegree reaches 0, enqueue it

```python
from collections import deque, defaultdict
def topo_sort(n, edges):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0:
                queue.append(nei)
    return order if len(order) == n else []  # [] means cycle
```

## Algorithm 2: DFS Post-order

```python
def topo_sort_dfs(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
    visited = [0] * n   # 0=unvisited, 1=in-stack, 2=done
    result = []
    has_cycle = [False]

    def dfs(node):
        if visited[node] == 1:
            has_cycle[0] = True; return
        if visited[node] == 2:
            return
        visited[node] = 1
        for nei in graph[node]:
            dfs(nei)
        visited[node] = 2
        result.append(node)

    for i in range(n):
        if visited[i] == 0:
            dfs(i)
    return [] if has_cycle[0] else result[::-1]
```

---

## Complexity

| Algorithm | Time | Space | Notes |
|-----------|------|-------|-------|
| Kahn's (BFS) | O(V+E) | O(V+E) | Intuitive, easy cycle detection |
| DFS Post-order | O(V+E) | O(V+E) | Recursive; watch stack depth |

---

## When to Use

| Signal in problem | Pattern |
|-------------------|---------|
| "prerequisite", "dependency" | Topological sort |
| "order of tasks" | Topological sort |
| "cycle in directed graph" | Topological sort + cycle check |
| "longest path in DAG" | Topo sort + DP on order |


In [ ]:
from collections import deque, defaultdict
from typing import List, Optional

def topo_sort(n, edges):
    """Kahn's algorithm. Returns [] if cycle detected."""
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0:
                queue.append(nei)
    return order if len(order) == n else []

print("Helpers loaded.")

---
## Easy Problems (20)

### E1. Course Schedule (LC 207)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given `numCourses` and `prerequisites`, return `True` if you can finish all courses.

**Approach:** Build DAG; run Kahn's. If topo order length == numCourses → no cycle → can finish.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def canFinish(numCourses, prerequisites):
    graph = defaultdict(list)
    indegree = [0] * numCourses
    for a, b in prerequisites:
        graph[b].append(a)
        indegree[a] += 1
    queue = deque(i for i in range(numCourses) if indegree[i] == 0)
    count = 0
    while queue:
        node = queue.popleft()
        count += 1
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0:
                queue.append(nei)
    return count == numCourses

assert canFinish(2, [[1,0]]) == True
assert canFinish(2, [[1,0],[0,1]]) == False
assert canFinish(1, []) == True
assert canFinish(3, [[1,0],[2,1]]) == True
print("All tests passed!")

### E2. Course Schedule II (LC 210)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Return the ordering of courses to finish all; return [] if impossible.

**Approach:** Kahn's topo sort; collect the order.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def findOrder(numCourses, prerequisites):
    graph = defaultdict(list)
    indegree = [0] * numCourses
    for a, b in prerequisites:
        graph[b].append(a)
        indegree[a] += 1
    queue = deque(i for i in range(numCourses) if indegree[i] == 0)
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0:
                queue.append(nei)
    return order if len(order) == numCourses else []

assert findOrder(2, [[1,0]]) == [0,1]
assert findOrder(2, [[0,1],[1,0]]) == []
assert len(findOrder(4, [[1,0],[2,0],[3,1],[3,2]])) == 4
print("All tests passed!")

### E3. Find Eventual Safe States (LC 802)

> 🏢 **Asked by:** Amazon, Google
Return all safe nodes (eventually lead to terminal node, no cycle).

**Approach:** Reverse edges → nodes that can reach a terminal become sources. Run topo sort on reversed graph.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def eventualSafeNodes(graph_input):
    n = len(graph_input)
    rev = defaultdict(list)
    indegree = [0] * n
    for u, neighbors in enumerate(graph_input):
        for v in neighbors:
            rev[v].append(u)
            indegree[u] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    safe = []
    while queue:
        node = queue.popleft()
        safe.append(node)
        for nei in rev[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0:
                queue.append(nei)
    return sorted(safe)

assert eventualSafeNodes([[1,2],[2,3],[5],[0],[5],[],[]]) == [2,4,5,6]
assert eventualSafeNodes([[1,2,3,4],[1,2],[3,4],[0,4],[]]) == [4]
print("All tests passed!")

### E4. Minimum Height Trees (LC 310)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Find all roots that minimize the height of the resulting tree.

**Approach:** Iteratively remove leaf nodes (degree 1) like peeling onion layers; last 1-2 nodes are roots.
**Time:** O(V) | **Space:** O(V)

In [ ]:
def findMinHeightTrees(n, edges):
    if n == 1: return [0]
    adj = defaultdict(set)
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    leaves = deque(i for i in range(n) if len(adj[i]) == 1)
    remaining = n
    while remaining > 2:
        remaining -= len(leaves)
        new_leaves = deque()
        while leaves:
            leaf = leaves.popleft()
            nei = adj[leaf].pop()
            adj[nei].remove(leaf)
            if len(adj[nei]) == 1:
                new_leaves.append(nei)
        leaves = new_leaves
    return list(leaves)

assert set(findMinHeightTrees(4, [[1,0],[1,2],[1,3]])) == {1}
assert set(findMinHeightTrees(6, [[3,0],[3,1],[3,2],[3,4],[5,4]])) == {3,4}
assert findMinHeightTrees(1, []) == [0]
print("All tests passed!")

### E5. Find All Possible Recipes from Given Supplies (LC 2115)

> 🏢 **Asked by:** Amazon, Google
Given recipes, ingredients, and supplies, return all recipes you can create.

**Approach:** Build dependency graph; initial supplies have indegree 0. Run topo sort; collect recipe nodes reached.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def findAllRecipes(recipes, ingredients, supplies):
    indegree = defaultdict(int)
    graph = defaultdict(list)
    recipe_set = set(recipes)
    for recipe, ing_list in zip(recipes, ingredients):
        for ing in ing_list:
            graph[ing].append(recipe)
            indegree[recipe] += 1
    queue = deque(supplies)
    result = []
    while queue:
        item = queue.popleft()
        if item in recipe_set:
            result.append(item)
        for nxt in graph[item]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)
    return result

assert set(findAllRecipes(["bread"],[["yeast","flour"]],["yeast","flour","corn"])) == {"bread"}
assert set(findAllRecipes(["bread","sandwich"],[["yeast","flour"],["bread","meat"]],["yeast","flour","meat"])) == {"bread","sandwich"}
print("All tests passed!")

### E6. Loud and Rich (LC 851)

> 🏢 **Asked by:** Amazon, Google
For each person, find the least quiet person among all richer-or-equal persons.

**Approach:** Build graph richer→poorer. Topo sort from richest; propagate the quietest answer down.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def loudAndRich(richer, quiet):
    n = len(quiet)
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in richer:
        graph[u].append(v)
        indegree[v] += 1
    answer = list(range(n))
    queue = deque(i for i in range(n) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            if quiet[answer[node]] < quiet[answer[nei]]:
                answer[nei] = answer[node]
            indegree[nei] -= 1
            if indegree[nei] == 0:
                queue.append(nei)
    return answer

assert loudAndRich([[1,0],[2,1],[3,1],[3,7],[4,3],[5,3],[6,3]],[3,2,5,4,6,1,7,0]) == [5,5,2,5,4,5,6,7]
print("All tests passed!")

### E7. Parallel Courses (LC 1136)

> 🏢 **Asked by:** Amazon, Google
Return minimum semesters to complete all courses (take all available each semester).

**Approach:** BFS layer-by-layer topo sort; each BFS level = one semester.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def minimumSemesters(n, relations):
    graph = defaultdict(list)
    indegree = [0] * (n+1)
    for u, v in relations:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(i for i in range(1, n+1) if indegree[i] == 0)
    semesters = 0; studied = 0
    while queue:
        semesters += 1
        for _ in range(len(queue)):
            node = queue.popleft()
            studied += 1
            for nei in graph[node]:
                indegree[nei] -= 1
                if indegree[nei] == 0:
                    queue.append(nei)
    return semesters if studied == n else -1

assert minimumSemesters(3, [[1,3],[2,3]]) == 2
assert minimumSemesters(3, [[1,2],[2,3],[3,1]]) == -1
print("All tests passed!")

### E8. Sequence Reconstruction (LC 444)

> 🏢 **Asked by:** Amazon, Google
Check if `org` is the only shortest supersequence reconstructible from `seqs`.

**Approach:** Build ordering constraints from seqs; topo sort must yield exactly one unique order matching org.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def sequenceReconstruction(org, seqs):
    n = len(org)
    graph = defaultdict(set)
    indegree = defaultdict(int)
    nodes = set()
    for seq in seqs:
        for x in seq: nodes.add(x)
        for i in range(len(seq)-1):
            u, v = seq[i], seq[i+1]
            if v not in graph[u]:
                graph[u].add(v)
                indegree[v] += 1
    for x in nodes:
        if x not in indegree: indegree[x] = 0
    if nodes != set(org): return False
    queue = deque(x for x in org if indegree[x] == 0)
    idx = 0
    while queue:
        if len(queue) > 1: return False
        node = queue.popleft()
        if idx >= len(org) or org[idx] != node: return False
        idx += 1
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return idx == n

assert sequenceReconstruction([1,2,3], [[1,2],[1,3],[2,3]]) == True
assert sequenceReconstruction([1,2,3], [[1,2]]) == False
print("All tests passed!")

### E9. Check if Graph is a DAG

> 🏢 **Asked by:** Amazon, Google
Given n nodes and directed edges, determine if the graph is a DAG (no cycle).

**Approach:** Run Kahn's topo sort; if all nodes are processed, no cycle exists → DAG.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def isDAG(n, edges):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    count = 0
    while queue:
        node = queue.popleft()
        count += 1
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return count == n

assert isDAG(3, [[0,1],[1,2]]) == True
assert isDAG(3, [[0,1],[1,2],[2,0]]) == False
assert isDAG(4, [[0,1],[0,2],[1,3],[2,3]]) == True
print("All tests passed!")

### E10. Count Nodes with No Incoming Edges (Sources)

> 🏢 **Asked by:** Amazon, Google
Given n nodes and edges, count nodes with indegree 0.

**Approach:** Build indegree array; count zeros.
**Time:** O(V+E) | **Space:** O(V)

In [ ]:
def countSources(n, edges):
    indegree = [0] * n
    for u, v in edges:
        indegree[v] += 1
    return sum(1 for x in indegree if x == 0)

assert countSources(4, [[0,1],[0,2],[1,3]]) == 1
assert countSources(3, [[0,1],[2,1]]) == 2
assert countSources(3, [[0,1],[1,2],[2,0]]) == 0
print("All tests passed!")

### E11. Count Nodes with No Outgoing Edges (Sinks)

> 🏢 **Asked by:** Amazon, Google
Given n nodes and edges, count nodes with outdegree 0.

**Approach:** Build outdegree array; count zeros.
**Time:** O(V+E) | **Space:** O(V)

In [ ]:
def countSinks(n, edges):
    outdegree = [0] * n
    for u, v in edges:
        outdegree[u] += 1
    return sum(1 for x in outdegree if x == 0)

assert countSinks(4, [[0,1],[0,2],[1,3]]) == 2
assert countSinks(3, [[0,1],[2,1]]) == 1
assert countSinks(3, [[0,1],[1,2],[2,0]]) == 0
print("All tests passed!")

### E12. Find All Nodes Reachable from Source in DAG

> 🏢 **Asked by:** Amazon, Google
Return all nodes reachable from `source` in a DAG.

**Approach:** BFS from source following directed edges.
**Time:** O(V+E) | **Space:** O(V)

In [ ]:
def reachableFromSource(n, edges, source):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
    visited = set()
    queue = deque([source])
    while queue:
        node = queue.popleft()
        if node in visited: continue
        visited.add(node)
        for nei in graph[node]:
            queue.append(nei)
    return sorted(visited)

assert reachableFromSource(5, [[0,1],[0,2],[1,3],[2,3],[3,4]], 0) == [0,1,2,3,4]
assert reachableFromSource(5, [[0,1],[0,2],[1,3],[2,3],[3,4]], 2) == [2,3,4]
print("All tests passed!")

### E13. Longest Path in DAG

> 🏢 **Asked by:** Amazon, Google
Find the length of the longest path in a DAG with n nodes.

**Approach:** Topo sort then DP: dp[v] = max(dp[u]+1) for all edges u→v.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def longestPathInDAG(n, edges):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    dp = [0] * n
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            dp[nei] = max(dp[nei], dp[node] + 1)
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return max(dp)

assert longestPathInDAG(4, [[0,1],[0,2],[1,3],[2,3]]) == 2
assert longestPathInDAG(3, [[0,1],[1,2]]) == 2
assert longestPathInDAG(1, []) == 0
print("All tests passed!")

### E14. Topological Sort with DFS Coloring

> 🏢 **Asked by:** Amazon, Google
Perform topological sort using DFS with 3-color marking (WHITE/GRAY/BLACK).

**Approach:** DFS: mark in-progress as GRAY, done as BLACK. GRAY revisit = cycle.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def topoSortDFSColor(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * n
    result = []
    cycle = [False]
    def dfs(u):
        if cycle[0]: return
        color[u] = GRAY
        for v in graph[u]:
            if color[v] == GRAY: cycle[0] = True; return
            if color[v] == WHITE: dfs(v)
        color[u] = BLACK
        result.append(u)
    for i in range(n):
        if color[i] == WHITE: dfs(i)
    return [] if cycle[0] else result[::-1]

order = topoSortDFSColor(4, [[0,1],[0,2],[1,3],[2,3]])
assert order.index(0) < order.index(3)
assert topoSortDFSColor(3, [[0,1],[1,2],[2,0]]) == []
print("All tests passed!")

### E15. Find All Ancestors of a Node in a DAG (LC 2192)

> 🏢 **Asked by:** Amazon, Google
Return list of ancestors for each node in sorted order.

**Approach:** Reverse edges → build reverse graph. BFS on reverse graph for each node.
**Time:** O(V*(V+E)) | **Space:** O(V+E)

In [ ]:
def getAncestors(n, edges):
    rev = defaultdict(list)
    for u, v in edges:
        rev[v].append(u)
    def bfs_ancestors(start):
        visited = set()
        queue = deque([start])
        while queue:
            node = queue.popleft()
            for par in rev[node]:
                if par not in visited:
                    visited.add(par)
                    queue.append(par)
        return sorted(visited)
    return [bfs_ancestors(i) for i in range(n)]

res = getAncestors(8, [[0,3],[0,4],[1,3],[2,4],[2,7],[3,5],[3,6],[3,7],[4,6]])
assert res[6] == [0,1,2,3,4]
assert res[3] == [0,1]
print("All tests passed!")

### E16. Build Dependency Ordering for Packages

> 🏢 **Asked by:** Amazon, Google
Given packages and their dependencies, return a valid install order.

**Approach:** Model as DAG (dependency → package). Kahn's topo sort gives install order.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def packageInstallOrder(packages, deps):
    all_nodes = set(packages)
    for u, v in deps:
        all_nodes.add(u); all_nodes.add(v)
    node_list = list(all_nodes)
    graph = defaultdict(list)
    indegree = {n: 0 for n in node_list}
    for u, v in deps:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(n for n in node_list if indegree[n] == 0)
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return order if len(order) == len(all_nodes) else []

order = packageInstallOrder(["a","b","c"], [["a","b"],["b","c"]])
assert order.index("a") < order.index("b") < order.index("c")
print("All tests passed!")

### E17. Find Nodes That Can Reach All Others in DAG

> 🏢 **Asked by:** Amazon, Google
Return all nodes from which every other node is reachable.

**Approach:** BFS from each node; keep those that reach all n nodes.
**Time:** O(V*(V+E)) | **Space:** O(V+E)

In [ ]:
def nodesReachAll(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
    def can_reach_all(start):
        visited = set()
        queue = deque([start])
        while queue:
            node = queue.popleft()
            if node in visited: continue
            visited.add(node)
            for nei in graph[node]: queue.append(nei)
        return len(visited) == n
    return [i for i in range(n) if can_reach_all(i)]

assert nodesReachAll(4, [[0,1],[0,2],[1,3],[2,3]]) == [0]
assert nodesReachAll(3, [[0,1],[0,2]]) == [0]
print("All tests passed!")

### E18. Largest Number (LC 179)

> 🏢 **Asked by:** Amazon, Google
Arrange numbers to form the largest number.

**Approach:** Custom comparator: compare str(a)+str(b) vs str(b)+str(a). Sort descending.
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import functools

def largestNumber(nums):
    def cmp(a, b):
        if a+b > b+a: return -1
        elif a+b < b+a: return 1
        return 0
    strs = [str(n) for n in nums]
    strs.sort(key=functools.cmp_to_key(cmp))
    result = "".join(strs)
    return "0" if result[0] == "0" else result

assert largestNumber([10,2]) == "210"
assert largestNumber([3,30,34,5,9]) == "9534330"
assert largestNumber([0,0]) == "0"
print("All tests passed!")

### E19. Sort Array by Increasing Frequency (LC 1636)

> 🏢 **Asked by:** Amazon, Google
Sort array by increasing frequency; ties broken by decreasing value.

**Approach:** Count frequencies with Counter; sort by (freq, -val).
**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from collections import Counter

def frequencySort(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], -x))

assert frequencySort([1,1,2,2,2,3]) == [3,1,1,2,2,2]
assert frequencySort([2,3,1,3,2]) == [1,3,3,2,2]
assert frequencySort([-1,1,-6,4,5,-6,1,4,1]) == [5,-1,4,4,-6,-6,1,1,1]
print("All tests passed!")

### E20. Find the Winner of the Circular Game (LC 1823)

> 🏢 **Asked by:** Amazon, Google
In a circle of n friends, every k-th person is eliminated. Find the winner.

**Approach:** Josephus problem. DP: f(1)=0, f(n)=(f(n-1)+k) % n. Answer is f(n)+1.
**Time:** O(n) | **Space:** O(1)

In [ ]:
def findTheWinner(n, k):
    pos = 0
    for i in range(2, n+1):
        pos = (pos + k) % i
    return pos + 1

assert findTheWinner(5, 2) == 3
assert findTheWinner(6, 5) == 1
assert findTheWinner(1, 1) == 1
print("All tests passed!")

---
## Medium Problems (15)

### M1. Alien Dictionary (LC 269)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Derive the character order from a sorted alien dictionary word list.

**Approach:** Compare adjacent words to extract ordering constraints; topo sort the chars.
**Time:** O(C) where C = total chars | **Space:** O(1) (26 letters)

In [ ]:
def alienOrder(words):
    graph = {c: set() for w in words for c in w}
    indegree = {c: 0 for c in graph}
    for i in range(len(words)-1):
        w1, w2 = words[i], words[i+1]
        min_len = min(len(w1), len(w2))
        if len(w1) > len(w2) and w1[:min_len] == w2[:min_len]:
            return ""
        for j in range(min_len):
            if w1[j] != w2[j]:
                if w2[j] not in graph[w1[j]]:
                    graph[w1[j]].add(w2[j])
                    indegree[w2[j]] += 1
                break
    queue = deque(c for c in indegree if indegree[c] == 0)
    result = []
    while queue:
        c = queue.popleft()
        result.append(c)
        for nei in graph[c]:
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return "".join(result) if len(result) == len(indegree) else ""

assert alienOrder(["wrt","wrf","er","ett","rftt"]) == "wertf"
assert alienOrder(["z","x"]) == "zx"
assert alienOrder(["z","x","z"]) == ""
print("All tests passed!")

### M2. Build a Matrix With Conditions (LC 2392)

> 🏢 **Asked by:** Amazon, Google
Build a k×k matrix placing 1..k such that row/col conditions are satisfied.

**Approach:** Topo sort rowConditions and colConditions separately; place numbers by their topo position.
**Time:** O(k+E) | **Space:** O(k+E)

In [ ]:
def buildMatrix(k, rowConditions, colConditions):
    def topo(n, edges):
        graph = defaultdict(list)
        indegree = [0] * (n+1)
        for u, v in edges:
            graph[u].append(v)
            indegree[v] += 1
        queue = deque(i for i in range(1, n+1) if indegree[i] == 0)
        order = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for nei in graph[node]:
                indegree[nei] -= 1
                if indegree[nei] == 0: queue.append(nei)
        return order if len(order) == n else []
    row_order = topo(k, rowConditions)
    col_order = topo(k, colConditions)
    if not row_order or not col_order: return []
    row_pos = {v: i for i, v in enumerate(row_order)}
    col_pos = {v: i for i, v in enumerate(col_order)}
    matrix = [[0]*k for _ in range(k)]
    for num in range(1, k+1):
        matrix[row_pos[num]][col_pos[num]] = num
    return matrix

res = buildMatrix(3, [[1,2],[3,2]], [[2,1],[3,2]])
assert res != []
assert buildMatrix(3, [[1,2],[2,3],[3,1]], [[1,2]]) == []
print("All tests passed!")

### M3. Longest Increasing Path in a Matrix (LC 329)

> 🏢 **Asked by:** Amazon, Google
Find the length of the longest strictly increasing path in a matrix.

**Approach:** Treat as DAG (smaller→larger), topo sort by value, dp on each cell.
**Time:** O(mn) | **Space:** O(mn)

In [ ]:
def longestIncreasingPath(matrix):
    if not matrix: return 0
    m, n = len(matrix), len(matrix[0])
    indegree = [[0]*n for _ in range(m)]
    dirs = [(0,1),(0,-1),(1,0),(-1,0)]
    for r in range(m):
        for c in range(n):
            for dr, dc in dirs:
                nr, nc = r+dr, c+dc
                if 0<=nr<m and 0<=nc<n and matrix[nr][nc] > matrix[r][c]:
                    indegree[nr][nc] += 1
    queue = deque((r,c) for r in range(m) for c in range(n) if indegree[r][c]==0)
    length = 0
    while queue:
        length += 1
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in dirs:
                nr, nc = r+dr, c+dc
                if 0<=nr<m and 0<=nc<n and matrix[nr][nc] > matrix[r][c]:
                    indegree[nr][nc] -= 1
                    if indegree[nr][nc] == 0: queue.append((nr, nc))
    return length

assert longestIncreasingPath([[9,9,4],[6,6,8],[2,1,1]]) == 4
assert longestIncreasingPath([[3,4,5],[3,2,6],[2,2,1]]) == 4
assert longestIncreasingPath([[1]]) == 1
print("All tests passed!")

### M4. Course Schedule IV (LC 1462)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
For each query [u,v], determine if u is a prerequisite of v.

**Approach:** Build reachability matrix via topo sort + DP propagation.
**Time:** O(V²+E) | **Space:** O(V²)

In [ ]:
def checkIfPrerequisite(numCourses, prerequisites, queries):
    # prerequisites[i] = [a, b]: must take a before b (a is prereq of b, edge a->b)
    graph = defaultdict(list)
    indegree = [0] * numCourses
    reach = [[False]*numCourses for _ in range(numCourses)]
    for a, b in prerequisites:   # a is prereq of b
        graph[a].append(b)
        indegree[b] += 1
        reach[a][b] = True
    queue = deque(i for i in range(numCourses) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            # Propagate: everything that can reach 'node' can also reach 'nei'
            for k in range(numCourses):
                if reach[k][node]:
                    reach[k][nei] = True
            reach[node][nei] = True
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return [reach[u][v] for u, v in queries]

assert checkIfPrerequisite(2, [[1,0]], [[0,1],[1,0]]) == [False, True]
assert checkIfPrerequisite(3, [[1,2],[1,0],[2,0]], [[1,0],[1,2]]) == [True, True]
print("All tests passed!")

### M5. Parallel Courses III (LC 2050)

> 🏢 **Asked by:** Amazon, Google
Minimum months to complete all courses where each has a duration and prerequisites.

**Approach:** Topo sort + DP. dp[v] = time[v] + max(dp[prereq]) propagated through DAG.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def minimumTime(n, relations, time):
    graph = defaultdict(list)
    indegree = [0] * (n+1)
    for u, v in relations:
        graph[u].append(v)
        indegree[v] += 1
    dp = [0] * (n+1)
    for i in range(1, n+1): dp[i] = time[i-1]
    queue = deque(i for i in range(1, n+1) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            dp[nei] = max(dp[nei], dp[node] + time[nei-1])
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return max(dp[1:])

assert minimumTime(3, [[1,3],[2,3]], [3,2,5]) == 8
assert minimumTime(5, [[1,5],[2,5],[3,5],[3,4],[4,5]], [1,2,3,4,5]) == 12
print("All tests passed!")

### M6. Reconstruct Itinerary (LC 332)

> 🏢 **Asked by:** Amazon, Google, Meta
Find itinerary using all tickets starting from JFK in lexicographic order.

**Approach:** Hierholzer's algorithm for Eulerian path: DFS, post-order append, reverse.
**Time:** O(E log E) | **Space:** O(E)

In [ ]:
def findItinerary(tickets):
    graph = defaultdict(list)
    for src, dst in sorted(tickets, reverse=True):
        graph[src].append(dst)
    result = []
    def dfs(airport):
        while graph[airport]:
            dfs(graph[airport].pop())
        result.append(airport)
    dfs("JFK")
    return result[::-1]

assert findItinerary([["MUC","LHR"],["JFK","MUC"],["SFO","SJC"],["LHR","SFO"]]) == ["JFK","MUC","LHR","SFO","SJC"]
assert findItinerary([["JFK","SFO"],["JFK","ATL"],["SFO","ATL"],["ATL","JFK"],["ATL","SFO"]]) == ["JFK","ATL","JFK","SFO","ATL","SFO"]
print("All tests passed!")

### M7. Time Needed to Inform All Employees (LC 1376)

> 🏢 **Asked by:** Amazon, Google
Find time for news to reach all employees (each manager informs subordinates simultaneously).

**Approach:** BFS from headID; propagate time[node] + informTime[node] to children.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def numOfMinutes(n, headID, manager, informTime):
    subordinates = defaultdict(list)
    for emp, mgr in enumerate(manager):
        if mgr != -1: subordinates[mgr].append(emp)
    queue = deque([(headID, 0)])
    max_time = 0
    while queue:
        emp, t = queue.popleft()
        max_time = max(max_time, t)
        for sub in subordinates[emp]:
            queue.append((sub, t + informTime[emp]))
    return max_time

assert numOfMinutes(1, 0, [-1], [0]) == 0
assert numOfMinutes(6, 2, [2,2,-1,2,2,2], [0,0,1,0,0,0]) == 1
assert numOfMinutes(7, 6, [1,2,3,4,5,6,-1], [0,6,5,4,3,2,1]) == 21
print("All tests passed!")

### M8. All Ancestors of a Node in a DAG — Propagation (LC 2192)

> 🏢 **Asked by:** Amazon, Google
For each node, return sorted list of all ancestors using topo sort propagation.

**Approach:** Topo sort; propagate ancestors forward: ancestors[v] = union(ancestors[u] ∪ {u}) for each u→v.
**Time:** O(V²) | **Space:** O(V²)

In [ ]:
def getAncestorsFull(n, edges):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    ancestors = [set() for _ in range(n)]
    queue = deque(i for i in range(n) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            ancestors[nei] |= ancestors[node] | {node}
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return [sorted(a) for a in ancestors]

res = getAncestorsFull(8, [[0,3],[0,4],[1,3],[2,4],[2,7],[3,5],[3,6],[3,7],[4,6]])
assert res[6] == [0,1,2,3,4]
assert res[7] == [0,1,2,3]
print("All tests passed!")

### M9. Sort Items by Groups Respecting Dependencies (LC 1203)

> 🏢 **Asked by:** Amazon, Google, Meta
Sort items where each has a group; inter/intra group ordering constraints must be respected.

**Approach:** Two-level topo sort: within each group AND between groups.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def sortItems(n, m, group, beforeItems):
    group = list(group)
    new_g = m
    for i in range(n):
        if group[i] == -1:
            group[i] = new_g; new_g += 1
    total_groups = new_g
    item_graph = defaultdict(list); item_deg = [0]*n
    grp_graph  = defaultdict(list); grp_deg   = [0]*total_groups
    grp_edges_added = defaultdict(set)
    for v in range(n):
        for u in beforeItems[v]:
            item_graph[u].append(v); item_deg[v] += 1
            gu, gv = group[u], group[v]
            if gu != gv and gv not in grp_edges_added[gu]:
                grp_edges_added[gu].add(gv)
                grp_graph[gu].append(gv); grp_deg[gv] += 1
    def topo_list(graph, indeg, nodes):
        q = deque(x for x in nodes if indeg[x] == 0)
        res = []
        while q:
            x = q.popleft(); res.append(x)
            for nb in graph[x]:
                indeg[nb] -= 1
                if indeg[nb] == 0: q.append(nb)
        return res if len(res) == len(nodes) else []
    item_order = topo_list(item_graph, item_deg, list(range(n)))
    grp_order  = topo_list(grp_graph, grp_deg, list(range(total_groups)))
    if not item_order or not grp_order: return []
    grp_items = defaultdict(list)
    for item in item_order:
        grp_items[group[item]].append(item)
    return [item for g in grp_order for item in grp_items[g]]

res = sortItems(8, 2, [-1,-1,1,0,0,1,0,-1], [[],[6],[5],[6],[3,6],[],[],[]])
assert res != [] and len(res) == 8
print("All tests passed!")

### M10. Minimum Semesters with At Most K Courses Per Semester

> 🏢 **Asked by:** Amazon, Google
Min semesters taking at most k courses per semester with prerequisites.

**Approach:** BFS topo sort — each semester take up to k available courses (greedy).
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def minNumberOfSemesters(n, dependencies, k):
    graph = defaultdict(list)
    indegree = [0] * (n+1)
    for u, v in dependencies:
        graph[u].append(v)
        indegree[v] += 1
    available = sorted(i for i in range(1, n+1) if indegree[i] == 0)
    semesters = 0; done = 0
    while done < n:
        semesters += 1
        take = available[:k]
        available = available[k:]
        done += len(take)
        next_avail = []
        for node in take:
            for nei in graph[node]:
                indegree[nei] -= 1
                if indegree[nei] == 0: next_avail.append(nei)
        available = sorted(available + next_avail)
    return semesters

assert minNumberOfSemesters(4, [[2,1],[3,1],[1,4]], 2) == 3
assert minNumberOfSemesters(5, [[2,1],[3,1],[4,1],[1,5]], 2) == 4
print("All tests passed!")

### M11. Strange Printer II (LC 1591)

> 🏢 **Asked by:** Google, Amazon
Determine if a target grid can be printed using a strange printer (each turn fills rectangle with one color).

**Approach:** For each color find bounding rectangle. If another color appears inside that rectangle, it must be printed after. Topo sort; cycle means impossible.
**Time:** O(C²*mn) | **Space:** O(C²)

In [ ]:
def isPrintable(targetGrid):
    m, n = len(targetGrid), len(targetGrid[0])
    bounds = {}
    for r in range(m):
        for c in range(n):
            col = targetGrid[r][c]
            if col not in bounds: bounds[col] = [r,c,r,c]
            else:
                bounds[col][0] = min(bounds[col][0], r)
                bounds[col][1] = min(bounds[col][1], c)
                bounds[col][2] = max(bounds[col][2], r)
                bounds[col][3] = max(bounds[col][3], c)
    colors = list(bounds.keys())
    graph = defaultdict(set)
    indegree = {c: 0 for c in colors}
    for col in colors:
        r1,c1,r2,c2 = bounds[col]
        for r in range(r1, r2+1):
            for c in range(c1, c2+1):
                other = targetGrid[r][c]
                if other != col and other not in graph[col]:
                    graph[col].add(other)
                    indegree[other] += 1
    queue = deque(c for c in colors if indegree[c] == 0)
    done = 0
    while queue:
        c = queue.popleft(); done += 1
        for nei in graph[c]:
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return done == len(colors)

assert isPrintable([[1,1,1,1],[1,2,2,1],[1,2,2,1],[1,1,1,1]]) == True
assert isPrintable([[1,1,1],[3,1,3]]) == False
print("All tests passed!")

### M12. BFS Layer Count in a DAG

> 🏢 **Asked by:** Amazon, Google
Find the number of BFS layers in a course prerequisite DAG.

**Approach:** Standard BFS layer counting (each layer = one parallel batch).
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def courseLevels(n, edges):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    levels = 0
    while queue:
        levels += 1
        for _ in range(len(queue)):
            node = queue.popleft()
            for nei in graph[node]:
                indegree[nei] -= 1
                if indegree[nei] == 0: queue.append(nei)
    return levels

assert courseLevels(4, [[0,1],[0,2],[1,3],[2,3]]) == 3
assert courseLevels(3, [[0,1],[1,2]]) == 3
assert courseLevels(1, []) == 1
print("All tests passed!")

### M13. Count Paths from Source to Destination in DAG

> 🏢 **Asked by:** Amazon, Google
Count the number of distinct paths from source to destination.

**Approach:** Topo sort then DP: dp[v] += dp[u] for each edge u→v.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def countPaths(n, edges, src, dst):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    dp = [0] * n
    dp[src] = 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            dp[nei] += dp[node]
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return dp[dst]

assert countPaths(4, [[0,1],[0,2],[1,3],[2,3]], 0, 3) == 2
assert countPaths(3, [[0,1],[0,2],[1,2]], 0, 2) == 2
print("All tests passed!")

### M14. Widest Level in a DAG

> 🏢 **Asked by:** Amazon, Google
Find the maximum number of nodes at any level in a DAG.

**Approach:** BFS layer by layer; track max layer width.
**Time:** O(V+E) | **Space:** O(V)

In [ ]:
def maxWidth(n, edges):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    max_w = 0
    while queue:
        max_w = max(max_w, len(queue))
        for _ in range(len(queue)):
            node = queue.popleft()
            for nei in graph[node]:
                indegree[nei] -= 1
                if indegree[nei] == 0: queue.append(nei)
    return max_w

assert maxWidth(4, [[0,1],[0,2],[1,3],[2,3]]) == 2
assert maxWidth(5, [[0,1],[0,2],[0,3],[1,4],[2,4],[3,4]]) == 3
print("All tests passed!")

### M15. Critical Path — Earliest Finish Time

> 🏢 **Asked by:** Amazon, Google
Given tasks with durations and dependencies, find the earliest time all tasks finish.

**Approach:** Topo sort + DP earliest start: dp[v] = max(dp[u]+dur[u]) for all u→v. Answer = max(dp[i]+dur[i]).
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def criticalPathLength(n, edges, duration):
    graph = defaultdict(list)
    indegree = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    dp = [0] * n
    queue = deque(i for i in range(n) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            dp[nei] = max(dp[nei], dp[node] + duration[node])
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return max(dp[i] + duration[i] for i in range(n))

assert criticalPathLength(3, [[0,1],[0,2],[1,2]], [3,2,1]) == 6
assert criticalPathLength(2, [[0,1]], [5,3]) == 8
assert criticalPathLength(1, [], [4]) == 4
print("All tests passed!")

---
## Hard Problems (10)

### H1. Alien Dictionary — Full with All Edge Cases (LC 269)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Handle prefix conflicts, disconnected components, all topo-sort edge cases.

**Approach:** Full implementation with prefix-length check, cycle detection, and incomplete ordering detection.
**Time:** O(C) | **Space:** O(1)

In [ ]:
def alienOrderFull(words):
    graph = defaultdict(set)
    indegree = {c: 0 for w in words for c in w}
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        found = False
        for j in range(min(len(w1), len(w2))):
            if w1[j] != w2[j]:
                if w2[j] not in graph[w1[j]]:
                    graph[w1[j]].add(w2[j])
                    indegree[w2[j]] += 1
                found = True; break
        if not found and len(w1) > len(w2):
            return ""
    queue = deque(sorted(c for c in indegree if indegree[c] == 0))
    res = []
    while queue:
        c = queue.popleft(); res.append(c)
        for nb in sorted(graph[c]):
            indegree[nb] -= 1
            if indegree[nb] == 0: queue.append(nb)
    return "".join(res) if len(res) == len(indegree) else ""

assert alienOrderFull(["wrt","wrf","er","ett","rftt"]) == "wertf"
assert alienOrderFull(["z","x"]) == "zx"
assert alienOrderFull(["z","x","z"]) == ""
assert alienOrderFull(["abc","ab"]) == ""
assert alienOrderFull(["a","b","a"]) == ""
print("All tests passed!")

### H2. Sort Items by Groups Respecting Dependencies Full (LC 1203)

> 🏢 **Asked by:** Amazon, Google, Meta
Full implementation handling all edge cases including self-loops and multi-group constraints.

**Approach:** Two-level topo sort — first within groups, then between groups. Assign unique IDs to ungrouped items.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def sortItemsFull(n, m, group, beforeItems):
    group = list(group)
    next_id = m
    for i in range(n):
        if group[i] == -1:
            group[i] = next_id; next_id += 1
    total_g = next_id
    item_graph = defaultdict(list); item_deg = [0]*n
    grp_graph  = defaultdict(list); grp_deg   = [0]*total_g
    grp_edges_set = defaultdict(set)
    for v in range(n):
        for u in beforeItems[v]:
            item_graph[u].append(v); item_deg[v] += 1
            gu, gv = group[u], group[v]
            if gu != gv and gv not in grp_edges_set[gu]:
                grp_edges_set[gu].add(gv)
                grp_graph[gu].append(gv); grp_deg[gv] += 1
    def topo(nodes, graph, deg):
        q = deque(x for x in nodes if deg[x] == 0)
        res = []
        while q:
            x = q.popleft(); res.append(x)
            for nb in graph[x]:
                deg[nb] -= 1
                if deg[nb] == 0: q.append(nb)
        return res if len(res) == len(nodes) else []
    item_order = topo(list(range(n)), item_graph, item_deg)
    grp_order  = topo(list(range(total_g)), grp_graph, grp_deg)
    if not item_order or not grp_order: return []
    grp_items = defaultdict(list)
    for item in item_order:
        grp_items[group[item]].append(item)
    return [item for g in grp_order for item in grp_items[g]]

res = sortItemsFull(8, 2, [-1,-1,1,0,0,1,0,-1], [[],[6],[5],[6],[3,6],[],[],[]])
assert len(res) == 8 and res != []
print("All tests passed!")

### H3. Parallel Courses III Full (LC 2050)

> 🏢 **Asked by:** Amazon, Google
Minimum time to complete all courses with processing times and prerequisites.

**Approach:** Topo sort + DP. dp[v] = time[v] + max(dp[prereq]) propagated through DAG.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def minimumTimeFull(n, relations, time):
    graph = defaultdict(list)
    indegree = [0] * (n+1)
    for u, v in relations:
        graph[u].append(v)
        indegree[v] += 1
    dp = [0] * (n+1)
    for i in range(1, n+1): dp[i] = time[i-1]
    queue = deque(i for i in range(1, n+1) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        for nei in graph[node]:
            dp[nei] = max(dp[nei], dp[node] + time[nei-1])
            indegree[nei] -= 1
            if indegree[nei] == 0: queue.append(nei)
    return max(dp[1:])

assert minimumTimeFull(3, [[1,3],[2,3]], [3,2,5]) == 8
assert minimumTimeFull(5, [[1,5],[2,5],[3,5],[3,4],[4,5]], [1,2,3,4,5]) == 12
assert minimumTimeFull(1, [], [5]) == 5
print("All tests passed!")

### H4. Minimum Number of Semesters — Bitmask DP (LC 1494)

> 🏢 **Asked by:** Amazon, Google
Minimum semesters to take all n (≤15) courses with prerequisite bitmasks, at most k per semester.

**Approach:** Bitmask DP. State = set of completed courses. dp[mask] = min semesters to reach mask.
**Time:** O(2^n * n) | **Space:** O(2^n)

In [ ]:
def minNumberOfSemestersBitmask(n, dependencies, k):
    prereq = [0] * n
    for u, v in dependencies:
        prereq[v-1] |= (1 << (u-1))
    INF = float("inf")
    dp = [INF] * (1 << n)
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] == INF: continue
        can_take = [c for c in range(n)
                    if not (mask >> c & 1) and (prereq[c] & mask) == prereq[c]]
        ct_mask = 0
        for c in can_take: ct_mask |= (1 << c)
        sub = ct_mask
        while sub:
            if bin(sub).count("1") <= k:
                nxt = mask | sub
                dp[nxt] = min(dp[nxt], dp[mask] + 1)
            sub = (sub-1) & ct_mask
    return dp[(1 << n) - 1]

assert minNumberOfSemestersBitmask(4, [[2,1],[3,1],[1,4]], 2) == 3
assert minNumberOfSemestersBitmask(5, [[2,1],[3,1],[4,1],[1,5]], 2) == 4
print("All tests passed!")

### H5. Reconstruct Itinerary — Full Hierholzer (LC 332)

> 🏢 **Asked by:** Amazon, Google, Meta
Find lexicographically smallest Eulerian path in a directed multigraph starting from JFK.

**Approach:** Sort neighbors in reverse; iterative DFS Hierholzer: follow edges greedily, post-order append, reverse.
**Time:** O(E log E) | **Space:** O(E)

In [ ]:
def findItineraryFull(tickets):
    graph = defaultdict(list)
    for src, dst in sorted(tickets, reverse=True):
        graph[src].append(dst)
    result = []
    stack = ["JFK"]
    while stack:
        while graph[stack[-1]]:
            nxt = graph[stack[-1]].pop()
            stack.append(nxt)
        result.append(stack.pop())
    return result[::-1]

assert findItineraryFull([["MUC","LHR"],["JFK","MUC"],["SFO","SJC"],["LHR","SFO"]]) == ["JFK","MUC","LHR","SFO","SJC"]
assert findItineraryFull([["JFK","SFO"],["JFK","ATL"],["SFO","ATL"],["ATL","JFK"],["ATL","SFO"]]) == ["JFK","ATL","JFK","SFO","ATL","SFO"]
assert findItineraryFull([["JFK","KUL"],["JFK","NRT"],["NRT","JFK"]]) == ["JFK","NRT","JFK","KUL"]
print("All tests passed!")

### H6. Longest Path With Different Adjacent Characters (LC 2246)

> 🏢 **Asked by:** Amazon, Google
Find longest path in a tree where no two adjacent nodes share the same character.

**Approach:** Root tree at 0. DFS: for each node collect top-2 longest child paths (different char). Answer = max sum of top-2 + 1.
**Time:** O(V) | **Space:** O(V)

In [ ]:
def longestPathDiffChars(parent, s):
    n = len(parent)
    children = defaultdict(list)
    for i in range(1, n):
        children[parent[i]].append(i)
    ans = [1]
    def dfs(node):
        top2 = [0, 0]
        for child in children[node]:
            length = dfs(child)
            if s[child] != s[node]:
                if length > top2[0]: top2[1] = top2[0]; top2[0] = length
                elif length > top2[1]: top2[1] = length
        ans[0] = max(ans[0], top2[0] + top2[1] + 1)
        return top2[0] + 1
    dfs(0)
    return ans[0]

assert longestPathDiffChars([-1,0,0,1,1,2], "abacbe") == 3
assert longestPathDiffChars([-1,0,0,0], "aabc") == 3
print("All tests passed!")

### H7. Maximum Employees to Be Invited to a Meeting (LC 2127)

> 🏢 **Asked by:** Amazon, Google
Each employee has a favorite colleague; find max employees to seat at a round table.

**Approach:** Find cycles in functional graph (each node out-degree=1). Case1: large cycle (≥3). Case2: sum of mutual-pairs (2-cycles) + chains extending from each end.
**Time:** O(V) | **Space:** O(V)

In [ ]:
def maximumInvitations(favorite):
    n = len(favorite)
    indegree = [0] * n
    for f in favorite: indegree[f] += 1
    queue = deque(i for i in range(n) if indegree[i] == 0)
    depth = [1] * n
    while queue:
        node = queue.popleft()
        nxt = favorite[node]
        depth[nxt] = max(depth[nxt], depth[node] + 1)
        indegree[nxt] -= 1
        if indegree[nxt] == 0: queue.append(nxt)
    visited = [False] * n
    case1 = 0
    case2 = 0
    for i in range(n):
        if not visited[i] and indegree[i] > 0:
            cycle = []
            node = i
            while not visited[node]:
                visited[node] = True
                cycle.append(node)
                node = favorite[node]
            cycle_len = len(cycle)
            if cycle_len == 2:
                u, v = cycle[0], cycle[1]
                case2 += depth[u] + depth[v]
            else:
                case1 = max(case1, cycle_len)
    return max(case1, case2)

assert maximumInvitations([2,2,1,2]) == 3
assert maximumInvitations([1,2,0]) == 3
assert maximumInvitations([3,0,1,4,1]) == 4
print("All tests passed!")

### H8. Minimum Score of a Path Between Two Cities (LC 2492)

> 🏢 **Asked by:** Amazon, Google
Find minimum road distance on any path from city 1 to city n (path can revisit nodes).

**Approach:** BFS/DFS to find all nodes in the same connected component as node 1. Min edge in that component.
**Time:** O(V+E) | **Space:** O(V+E)

In [ ]:
def minScore(n, roads):
    adj = defaultdict(list)
    for u, v, d in roads:
        adj[u].append((v, d))
        adj[v].append((u, d))
    visited = set()
    min_dist = [float("inf")]
    queue = deque([1])
    visited.add(1)
    while queue:
        node = queue.popleft()
        for nei, d in adj[node]:
            min_dist[0] = min(min_dist[0], d)
            if nei not in visited:
                visited.add(nei)
                queue.append(nei)
    return min_dist[0]

assert minScore(4, [[1,2,9],[2,3,6],[2,4,5],[1,4,7]]) == 5
assert minScore(4, [[1,2,2],[1,3,4],[3,4,7]]) == 2
print("All tests passed!")

### H9. Find Critical and Pseudo-Critical Edges in MST (LC 1489)

> 🏢 **Asked by:** Amazon, Google
Find all critical and pseudo-critical edges in a minimum spanning tree.

**Approach:** For each edge: critical if removing it increases MST weight. Pseudo-critical if forcing it into MST doesn't increase weight.
**Time:** O(E² α(V)) | **Space:** O(V)

In [ ]:
def findCriticalAndPseudoCriticalEdges(n, edges):
    def find(p, x):
        while p[x] != x: p[x] = p[p[x]]; x = p[x]
        return x
    def union(p, rank, x, y):
        rx, ry = find(p,x), find(p,y)
        if rx == ry: return False
        if rank[rx] < rank[ry]: rx, ry = ry, rx
        p[ry] = rx
        if rank[rx] == rank[ry]: rank[rx] += 1
        return True
    indexed = sorted([(w,u,v,i) for i,(u,v,w) in enumerate(edges)])
    def mst_weight(skip=-1, force=-1):
        p = list(range(n)); rank = [0]*n
        w = 0; cnt = 0
        if force != -1:
            fu,fv,fw = edges[force]
            union(p, rank, fu, fv); w += fw; cnt += 1
        for fw,fu,fv,fi in indexed:
            if fi == skip: continue
            if union(p, rank, fu, fv):
                w += fw; cnt += 1
        return w if cnt == n-1 else float("inf")
    base = mst_weight()
    critical, pseudo = [], []
    for _,_,_,i in indexed:
        if mst_weight(skip=i) > base: critical.append(i)
        elif mst_weight(force=i) == base: pseudo.append(i)
    return [critical, pseudo]

res = findCriticalAndPseudoCriticalEdges(5, [[0,1,1],[1,2,1],[2,3,2],[0,3,2],[0,4,3],[3,4,3],[1,4,6]])
assert 0 in res[0] and 1 in res[0]  # edges 0 and 1 are critical (weight-1 edges)
res2 = findCriticalAndPseudoCriticalEdges(4, [[0,1,1],[1,2,1],[2,3,1],[0,3,1]])
assert res2[0] == []
print("All tests passed!")

### H10. Count Subtrees With Max Distance Between Cities (LC 1617)

> 🏢 **Asked by:** Amazon, Google
For each possible max distance d, count subsets of cities forming a connected subtree with that diameter.

**Approach:** Enumerate all subsets (n≤15); for each connected subset compute diameter via BFS; increment count for that diameter.
**Time:** O(2^n * n²) | **Space:** O(2^n)

In [ ]:
def countSubgraphsForEachDiameter(n, edges):
    adj = [[] for _ in range(n+1)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    def bfs(start, nodes_set):
        dist = {start: 0}
        q = deque([start])
        while q:
            node = q.popleft()
            for nei in adj[node]:
                if nei in nodes_set and nei not in dist:
                    dist[nei] = dist[node] + 1
                    q.append(nei)
        return dist
    ans = [0] * (n-1)
    for mask in range(1, 1<<n):
        nodes = [i+1 for i in range(n) if mask>>i&1]
        if len(nodes) < 2: continue
        nodes_set = set(nodes)
        d = bfs(nodes[0], nodes_set)
        if len(d) != len(nodes): continue
        diameter = max(d.values())
        if diameter > 0: ans[diameter-1] += 1
    return ans

assert countSubgraphsForEachDiameter(4, [[1,2],[2,3],[2,4]]) == [4,3,0]
assert countSubgraphsForEachDiameter(2, [[1,2]]) == [1]
print("All tests passed!")

## Easy Problems (21-40)

## 21. Find Order to Finish Tasks (Basic Topological Sort)

> 🏢 **Asked by:** Amazon, Google
Given n tasks (0..n-1) and a list of [prerequisite, task] pairs, return any valid order to finish all tasks. Return [] if impossible.

**Approach**: Kahn's algorithm (BFS with indegree). Start with 0-indegree nodes; process queue reducing neighbors' indegrees.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
from collections import deque, defaultdict

def findOrder(n, prerequisites):
    graph = defaultdict(list)
    indegree = [0]*n
    for task, pre in prerequisites:
        graph[pre].append(task); indegree[task]+=1
    queue = deque(i for i in range(n) if indegree[i]==0)
    order = []
    while queue:
        node = queue.popleft(); order.append(node)
        for nei in graph[node]:
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return order if len(order)==n else []

res = findOrder(4, [[1,0],[2,0],[3,1],[3,2]])
assert len(res)==4 and res.index(0)<res.index(1) and res.index(0)<res.index(2)
assert findOrder(2, [[1,0],[0,1]]) == []
print('All tests passed!')

## 22. Check if a Path Exists in DAG from Source to Destination

> 🏢 **Asked by:** Amazon, Google
Given a DAG with n nodes and directed edges, return True if there is a path from source to destination.

**Approach**: BFS from source following directed edges; return True if destination is visited.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def pathExistsDAG(n, edges, source, destination):
    if source==destination: return True
    graph = defaultdict(list)
    for u,v in edges: graph[u].append(v)
    visited={source}; queue=deque([source])
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            if nei==destination: return True
            if nei not in visited: visited.add(nei); queue.append(nei)
    return False

assert pathExistsDAG(4,[[0,1],[1,2],[2,3]],0,3) == True
assert pathExistsDAG(4,[[0,1],[2,3]],0,3) == False
print('All tests passed!')

## 23. Count Nodes with No Prerequisites

> 🏢 **Asked by:** Amazon, Google
Given n tasks and a list of prerequisite pairs, return the count of tasks that have no prerequisites.

**Approach**: Count nodes with indegree 0 after building the indegree array.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def countNoPrereqs(n, prerequisites):
    indegree=[0]*n
    for task, _ in prerequisites: indegree[task]+=1
    return sum(1 for i in range(n) if indegree[i]==0)

assert countNoPrereqs(4,[[1,0],[2,0],[3,1]]) == 1  # only task 0
assert countNoPrereqs(3,[]) == 3
assert countNoPrereqs(4,[[1,0],[2,1],[3,2]]) == 1
print('All tests passed!')

## 24. Find All Tasks That Must Be Done Before Task K

> 🏢 **Asked by:** Amazon, Google
Given n tasks and prerequisite pairs, return all tasks that must be completed before task K (all ancestors of K in the dependency DAG).

**Approach**: Build reverse graph (edges pointing to prerequisites); BFS/DFS backwards from K.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def prerequisitesOf(n, prerequisites, k):
    rev = defaultdict(list)
    for task, pre in prerequisites: rev[task].append(pre)
    visited=set(); queue=deque([k])
    while queue:
        node=queue.popleft()
        for pre in rev[node]:
            if pre not in visited:
                visited.add(pre); queue.append(pre)
    return sorted(visited)

assert prerequisitesOf(4,[[1,0],[2,1],[3,2]],3) == [0,1,2]
assert prerequisitesOf(4,[[1,0],[2,0],[3,1],[3,2]],3) == [0,1,2]
assert prerequisitesOf(3,[[1,0]],0) == []
print('All tests passed!')

## 25. Print All Topological Orderings (Small Graph)

> 🏢 **Asked by:** Amazon, Google
Given a small DAG, enumerate all valid topological orderings using backtracking.

**Approach**: At each step choose any 0-indegree node, add to current ordering, reduce neighbors' indegrees, recurse. Backtrack after each choice.

- **Time**: O(V! in worst case)
- **Space**: O(V)

In [ ]:
def allTopologicalOrders(n, edges):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    results=[]
    def backtrack(path, ind):
        if len(path)==n: results.append(list(path)); return
        for node in range(n):
            if ind[node]==0 and node not in path:
                path.append(node)
                for nei in graph[node]: ind[nei]-=1
                backtrack(path, ind)
                path.pop()
                for nei in graph[node]: ind[nei]+=1
    backtrack([], indegree[:])
    return results

orders = allTopologicalOrders(3,[[0,2],[1,2]])
assert len(orders)==2
assert all(o.index(0)<o.index(2) and o.index(1)<o.index(2) for o in orders)
print('All tests passed!')

## 26. Check if Tasks Can Be Completed in Time

> 🏢 **Asked by:** Amazon, Google
Given n tasks each with a duration and a list of dependencies, check if all tasks can be completed within a given total time limit (assuming parallel execution respecting dependencies).

**Approach**: Topological sort (Kahn's); compute the earliest completion time of each task. Return whether max completion time <= limit.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def canCompleteInTime(n, durations, dependencies, limit):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in dependencies: graph[u].append(v); indegree[v]+=1
    finish=[durations[i] for i in range(n)]
    queue=deque(i for i in range(n) if indegree[i]==0)
    processed=0
    while queue:
        node=queue.popleft(); processed+=1
        for nei in graph[node]:
            finish[nei]=max(finish[nei], finish[node]+durations[nei])
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    if processed!=n: return False  # cycle
    return max(finish)<=limit

# Tasks 0,1 in parallel (each take 3), then task 2 (takes 2) => total=5
assert canCompleteInTime(3,[3,3,2],[[0,2],[1,2]],5) == True
assert canCompleteInTime(3,[3,3,2],[[0,2],[1,2]],4) == False
print('All tests passed!')

## 27. Minimum Time to Finish All Jobs (Simplified Linear)

> 🏢 **Asked by:** Amazon, Google
Given tasks with durations and dependencies (chain), compute the minimum time to finish all tasks sequentially (critical path in a chain DAG).

**Approach**: Simple topological traversal (chain); accumulate duration along the chain.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def minTimeToFinish(n, durations, deps):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in deps: graph[u].append(v); indegree[v]+=1
    earliest=[durations[i] for i in range(n)]
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            earliest[nei]=max(earliest[nei], earliest[node]+durations[nei])
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return max(earliest)

# Chain 0->1->2 with durations 2,3,4 => 2+3+4=9
assert minTimeToFinish(3,[2,3,4],[[0,1],[1,2]]) == 9
# Parallel: 0->2 and 1->2; durations [3,5,1] => max(3,5)+1=6
assert minTimeToFinish(3,[3,5,1],[[0,2],[1,2]]) == 6
print('All tests passed!')

## 28. Find the Next Task After Completing Prerequisites

> 🏢 **Asked by:** Amazon, Google
Given a topological order, for a given completed task return the list of tasks that become immediately available (all their prerequisites now satisfied).

**Approach**: Track indegrees; decrement neighbors of completed task; return nodes that reach 0.

- **Time**: O(E)
- **Space**: O(V)

In [ ]:
def nextAvailableTasks(n, prerequisites, completed_task, current_indegree):
    graph=defaultdict(list)
    for task, pre in prerequisites: graph[pre].append(task)
    ind=list(current_indegree)
    newly_available=[]
    for nei in graph[completed_task]:
        ind[nei]-=1
        if ind[nei]==0: newly_available.append(nei)
    return sorted(newly_available)

# Tasks: 0->1, 0->2, 1->3, 2->3
# After completing 0, tasks 1 and 2 become available
ind=[0,1,1,2]
assert nextAvailableTasks(4,[[1,0],[2,0],[3,1],[3,2]],0,ind)==[1,2]
print('All tests passed!')

## 29. Count Total Prerequisites for Each Task

> 🏢 **Asked by:** Amazon, Google
Given n tasks and prerequisite pairs, return an array where result[i] is the total number of tasks that must be completed before task i (all ancestors).

**Approach**: For each node BFS/DFS backwards through the reverse dependency graph, counting all reachable predecessors.

- **Time**: O(V·(V+E))
- **Space**: O(V)

In [ ]:
def countAllPrereqs(n, prerequisites):
    rev=defaultdict(list)
    for task, pre in prerequisites: rev[task].append(pre)
    def count_ancestors(k):
        visited=set(); queue=deque([k])
        while queue:
            node=queue.popleft()
            for pre in rev[node]:
                if pre not in visited: visited.add(pre); queue.append(pre)
        return len(visited)
    return [count_ancestors(i) for i in range(n)]

assert countAllPrereqs(4,[[1,0],[2,1],[3,2]]) == [0,1,2,3]
assert countAllPrereqs(3,[]) == [0,0,0]
print('All tests passed!')

## 30. Build Order (Tasks With Dependencies)

> 🏢 **Asked by:** Amazon, Google
Classic 'build order' problem: given projects and dependency pairs (a,b) meaning b depends on a, return a valid build order or [] if impossible.

**Approach**: Kahn's topological sort. 0-indegree nodes can be built first; process queue reducing dependents' indegrees.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def buildOrder(projects, dependencies):
    graph=defaultdict(list); indegree={p:0 for p in projects}
    for a,b in dependencies: graph[a].append(b); indegree[b]+=1
    queue=deque(p for p in projects if indegree[p]==0)
    order=[]
    while queue:
        proj=queue.popleft(); order.append(proj)
        for dep in graph[proj]:
            indegree[dep]-=1
            if indegree[dep]==0: queue.append(dep)
    return order if len(order)==len(projects) else []

res=buildOrder(['a','b','c','d','e','f'],[('a','d'),('f','b'),('b','d'),('f','a'),('d','c')])
assert len(res)==6 and res.index('a')<res.index('d') and res.index('d')<res.index('c')
assert buildOrder(['a','b'],[('a','b'),('b','a')]) == []
print('All tests passed!')

## 31. Detect if Adding Edge Creates a Cycle

> 🏢 **Asked by:** Amazon, Google
Given a DAG with n nodes and existing directed edges, return True if adding a new edge (u, v) would create a cycle.

**Approach**: Check if v can already reach u in the existing graph. If yes, adding u->v creates a cycle.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def wouldCreateCycle(n, existing_edges, new_u, new_v):
    # Would adding new_u -> new_v create a cycle?
    # Cycle exists iff new_v can already reach new_u
    graph=defaultdict(list)
    for u,v in existing_edges: graph[u].append(v)
    visited={new_v}; queue=deque([new_v])
    while queue:
        node=queue.popleft()
        if node==new_u: return True
        for nei in graph[node]:
            if nei not in visited: visited.add(nei); queue.append(nei)
    return False

# 0->1->2->3; adding 3->0 creates cycle
assert wouldCreateCycle(4,[[0,1],[1,2],[2,3]],3,0) == True
# 0->1->2->3; adding 0->3 does NOT (just a shortcut)
assert wouldCreateCycle(4,[[0,1],[1,2],[2,3]],0,3) == False
print('All tests passed!')

## 32. Find Number of Tasks Reachable From Start

> 🏢 **Asked by:** Amazon, Google
Given a DAG and a start task, return the count of tasks reachable from start (following dependency edges).

**Approach**: BFS from start on the directed graph; count all visited nodes.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def countReachableFromStart(n, edges, start):
    graph=defaultdict(list)
    for u,v in edges: graph[u].append(v)
    visited={start}; queue=deque([start])
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            if nei not in visited: visited.add(nei); queue.append(nei)
    return len(visited)

assert countReachableFromStart(5,[[0,1],[0,2],[1,3],[2,4]],0) == 5
assert countReachableFromStart(5,[[0,1],[2,3]],0) == 2
print('All tests passed!')

## 33. Longest Chain of Tasks

> 🏢 **Asked by:** Amazon, Google
Given n tasks and dependency edges (u->v means u must come before v), find the length of the longest chain (critical path in terms of node count).

**Approach**: Topological sort + DP. dp[v] = max(dp[u]+1 for all predecessors u of v).

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def longestChain(n, edges):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    dp=[1]*n
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            dp[nei]=max(dp[nei], dp[node]+1)
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return max(dp)

assert longestChain(4,[[0,1],[1,2],[2,3]]) == 4
assert longestChain(4,[[0,2],[1,2],[2,3]]) == 3
assert longestChain(3,[]) == 1
print('All tests passed!')

## 34. Find All Independent Tasks (No Prerequisites)

> 🏢 **Asked by:** Amazon, Google
Given n tasks and prerequisite pairs, return the list of tasks that have no prerequisites at all.

**Approach**: Compute indegree of each node; return nodes with indegree 0.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def independentTasks(n, prerequisites):
    indegree=[0]*n
    for task,_ in prerequisites: indegree[task]+=1
    return [i for i in range(n) if indegree[i]==0]

assert independentTasks(4,[[1,0],[2,0],[3,1]]) == [0]
assert independentTasks(3,[]) == [0,1,2]
assert sorted(independentTasks(4,[[2,0],[3,1]])) == [0,1]
print('All tests passed!')

## 35. Task Dependency Level (Level in Topological Order)

> 🏢 **Asked by:** Amazon, Google
Given n tasks and dependencies, assign each task its 'level' (distance from any root in the DAG, i.e., the length of the longest prerequisite chain + 1).

**Approach**: Topological BFS; level[v] = max(level[u]+1) for all predecessors u.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def taskLevels(n, prerequisites):
    graph=defaultdict(list); indegree=[0]*n
    for task,pre in prerequisites: graph[pre].append(task); indegree[task]+=1
    level=[0]*n
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            level[nei]=max(level[nei], level[node]+1)
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return level

assert taskLevels(4,[[1,0],[2,1],[3,2]]) == [0,1,2,3]
assert taskLevels(4,[[2,0],[2,1],[3,2]]) == [0,0,1,2]
print('All tests passed!')

## 36. Count Tasks at Each Dependency Level

> 🏢 **Asked by:** Amazon, Google
Given n tasks and dependencies, return a list where result[k] is the number of tasks at level k.

**Approach**: Compute task levels (as in E35); use a Counter on the level values.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
from collections import Counter

def countTasksPerLevel(n, prerequisites):
    graph=defaultdict(list); indegree=[0]*n
    for task,pre in prerequisites: graph[pre].append(task); indegree[task]+=1
    level=[0]*n
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            level[nei]=max(level[nei],level[node]+1)
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    cnt=Counter(level)
    return [cnt[k] for k in range(max(level)+1)]

assert countTasksPerLevel(4,[[1,0],[2,1],[3,2]]) == [1,1,1,1]
assert countTasksPerLevel(5,[[2,0],[2,1],[3,2],[4,2]]) == [2,1,2]
print('All tests passed!')

## 37. Sort Tasks by Priority Respecting Dependencies

> 🏢 **Asked by:** Amazon, Google
Given tasks with priorities and dependencies, return tasks in topological order breaking ties by priority (lower value = higher priority first).

**Approach**: Kahn's algorithm with a min-heap instead of a plain queue to always pick the highest-priority (lowest-value) available task.

- **Time**: O((V+E) log V)
- **Space**: O(V + E)

In [ ]:
import heapq

def sortByPriority(n, priorities, prerequisites):
    graph=defaultdict(list); indegree=[0]*n
    for task,pre in prerequisites: graph[pre].append(task); indegree[task]+=1
    heap=[(priorities[i],i) for i in range(n) if indegree[i]==0]
    heapq.heapify(heap)
    order=[]
    while heap:
        _,node=heapq.heappop(heap); order.append(node)
        for nei in graph[node]:
            indegree[nei]-=1
            if indegree[nei]==0: heapq.heappush(heap,(priorities[nei],nei))
    return order if len(order)==n else []

# Tasks 0,1 both available; priority [5,1,3]; should pick 1 first
res=sortByPriority(3,[5,1,3],[[2,0],[2,1]])
assert res[0]==1 and res[1]==0 and res[2]==2
print('All tests passed!')

## 38. Find All Leaf Tasks (No Dependents)

> 🏢 **Asked by:** Amazon, Google
Given n tasks and dependency edges, return the list of tasks that nothing else depends on (out-degree 0, i.e., terminal tasks).

**Approach**: Build the graph; return nodes that appear as no edge's source (outdegree 0).

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def leafTasks(n, prerequisites):
    has_dependents=set(pre for _,pre in prerequisites)
    return [i for i in range(n) if i not in has_dependents]

assert sorted(leafTasks(4,[[1,0],[2,0],[3,1],[3,2]])) == [3]
assert sorted(leafTasks(3,[])) == [0,1,2]
assert sorted(leafTasks(4,[[1,0],[2,1]])) == [2,3]
print('All tests passed!')

## 39. Check if Two Tasks are in Dependency Order

> 🏢 **Asked by:** Amazon, Google
Given n tasks and dependencies, return True if task A must be completed before task B (i.e., A is an ancestor of B in the DAG).

**Approach**: BFS forward from A; check if B is reachable.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def isABeforeB(n, prerequisites, a, b):
    graph=defaultdict(list)
    for task,pre in prerequisites: graph[pre].append(task)
    visited={a}; queue=deque([a])
    while queue:
        node=queue.popleft()
        if node==b: return True
        for nei in graph[node]:
            if nei not in visited: visited.add(nei); queue.append(nei)
    return False

assert isABeforeB(4,[[1,0],[2,1],[3,2]],0,3) == True
assert isABeforeB(4,[[1,0],[2,1],[3,2]],2,0) == False
assert isABeforeB(4,[[1,0],[2,0]],1,2) == False
print('All tests passed!')

## 40. Minimum Tasks to Remove to Make DAG Acyclic

> 🏢 **Asked by:** Amazon, Google
Given a directed graph (possibly with cycles), return the minimum number of edges to remove to make it a DAG.

**Approach**: This is the Minimum Feedback Arc Set problem (NP-hard in general). For tournaments/small graphs use greedy: for each strongly connected component (SCC), the minimum edges to remove equals |SCC_edges| - |SCC_nodes| + 1 per SCC with cycles. Here we use a simplified greedy: count back-edges in DFS.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def minEdgesToRemoveForDAG(n, edges):
    # Count back-edges (edges that point to ancestors in DFS tree)
    graph=defaultdict(list)
    for u,v in edges: graph[u].append(v)
    color=[0]*n; back_edges=[0]
    def dfs(node):
        color[node]=1
        for nei in graph[node]:
            if color[nei]==1: back_edges[0]+=1
            elif color[nei]==0: dfs(nei)
        color[node]=2
    for i in range(n):
        if color[i]==0: dfs(i)
    return back_edges[0]

assert minEdgesToRemoveForDAG(4,[[0,1],[1,2],[2,3],[3,1]]) == 1
assert minEdgesToRemoveForDAG(3,[[0,1],[1,2],[2,0]]) == 1
assert minEdgesToRemoveForDAG(3,[[0,1],[1,2]]) == 0
print('All tests passed!')

## Medium Problems (16-30)

## M16. Course Schedule III (LC 630) — Greedy Scheduling

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given courses [duration, lastDay], find the maximum number of courses you can take. You must finish each course by its lastDay.

**Approach**: Sort by lastDay. Use a max-heap of durations. Greedily take each course; if current time exceeds lastDay, swap out the longest course taken so far.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
def scheduleCourse(courses):
    courses.sort(key=lambda x: x[1])
    heap=[]; time=0
    for duration, last in courses:
        heapq.heappush(heap, -duration); time+=duration
        if time>last:
            time+=heapq.heappop(heap)  # heappop returns negative
    return len(heap)

assert scheduleCourse([[100,200],[200,1300],[1000,1250],[2000,3200]]) == 3
assert scheduleCourse([[1,2]]) == 1
assert scheduleCourse([[3,2],[4,3]]) == 0
print('All tests passed!')

## M17. Minimum Height Trees (LC 310) — Full With Edge Cases

> 🏢 **Asked by:** Amazon, Google, Microsoft
A tree with n nodes has roots that minimize height. Find all such roots.

**Approach**: Iteratively trim leaf nodes (degree 1). The last 1 or 2 remaining nodes are the MHT roots.

- **Time**: O(n)
- **Space**: O(n)

In [ ]:
def findMinHeightTrees(n, edges):
    if n==1: return [0]
    graph=defaultdict(set)
    for u,v in edges: graph[u].add(v); graph[v].add(u)
    leaves=deque(i for i in range(n) if len(graph[i])==1)
    remaining=n
    while remaining>2:
        remaining-=len(leaves)
        new_leaves=deque()
        while leaves:
            leaf=leaves.popleft()
            nei=graph[leaf].pop()
            graph[nei].discard(leaf)
            if len(graph[nei])==1: new_leaves.append(nei)
        leaves=new_leaves
    return list(leaves)

assert sorted(findMinHeightTrees(4,[[1,0],[1,2],[1,3]])) == [1]
assert sorted(findMinHeightTrees(6,[[3,0],[3,1],[3,2],[3,4],[5,4]])) == [3,4]
assert findMinHeightTrees(1,[]) == [0]
print('All tests passed!')

## M18. Find All Ancestors of a Node in a DAG (LC 2192) — Full

> 🏢 **Asked by:** Amazon, Google
Given a DAG with n nodes, for each node return a sorted list of all its ancestors.

**Approach**: Topological sort; for each node propagate its ancestor set (union with its own index) to all descendants.

- **Time**: O(V²) worst case
- **Space**: O(V²)

In [ ]:
def getAncestors(n, edges):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    ancestors=[set() for _ in range(n)]
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            ancestors[nei].update(ancestors[node])
            ancestors[nei].add(node)
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return [sorted(a) for a in ancestors]

res=getAncestors(8,[[0,3],[0,4],[1,3],[2,4],[2,7],[3,5],[3,6],[3,7],[4,6]])
assert res[6]==[0,1,2,3,4] and res[7]==[0,1,2,3]
print('All tests passed!')

## M19. Parallel Courses (LC 1136) — Minimum Semesters

> 🏢 **Asked by:** Amazon, Google
Given n courses and prerequisite relations, find the minimum number of semesters to finish all courses (take all available courses each semester).

**Approach**: BFS layer-by-layer (Kahn's). Each BFS layer = one semester. Return layer count; if not all taken return -1.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def minimumSemesters(n, relations):
    graph=defaultdict(list); indegree=[0]*n
    for pre,course in relations: graph[pre-1].append(course-1); indegree[course-1]+=1
    queue=deque(i for i in range(n) if indegree[i]==0)
    semesters=taken=0
    while queue:
        semesters+=1
        for _ in range(len(queue)):
            node=queue.popleft(); taken+=1
            for nei in graph[node]:
                indegree[nei]-=1
                if indegree[nei]==0: queue.append(nei)
    return semesters if taken==n else -1

assert minimumSemesters(3,[[1,3],[2,3]]) == 2
assert minimumSemesters(3,[[1,2],[2,3],[3,1]]) == -1
print('All tests passed!')

## M20. Topological Sort — Lexicographically Smallest

> 🏢 **Asked by:** Amazon, Google
Given a DAG, return the lexicographically smallest topological ordering by always choosing the smallest-numbered available node.

**Approach**: Kahn's algorithm with a min-heap. Always pop the smallest-index 0-indegree node.

- **Time**: O((V+E) log V)
- **Space**: O(V + E)

In [ ]:
def lexSmallestTopo(n, edges):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    heap=list(i for i in range(n) if indegree[i]==0)
    heapq.heapify(heap)
    order=[]
    while heap:
        node=heapq.heappop(heap); order.append(node)
        for nei in graph[node]:
            indegree[nei]-=1
            if indegree[nei]==0: heapq.heappush(heap,nei)
    return order if len(order)==n else []

res=lexSmallestTopo(4,[[3,1],[3,2],[2,0]])
assert res==[3,1,2,0]
res2=lexSmallestTopo(4,[[0,1],[0,2],[1,3],[2,3]])
assert res2==[0,1,2,3]
print('All tests passed!')

## M21. Detect Cycle in Undirected Graph — DFS

> 🏢 **Asked by:** Amazon, Google
Given an undirected graph with n nodes and edges, return True if the graph contains a cycle.

**Approach**: DFS tracking parent. If a neighbor is already visited and is not the current parent, a cycle exists.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def hasCycleUndirected(n, edges):
    graph=defaultdict(list)
    for u,v in edges: graph[u].append(v); graph[v].append(u)
    visited=set()
    def dfs(node, parent):
        visited.add(node)
        for nei in graph[node]:
            if nei not in visited:
                if dfs(nei, node): return True
            elif nei!=parent: return True
        return False
    for i in range(n):
        if i not in visited:
            if dfs(i,-1): return True
    return False

assert hasCycleUndirected(4,[[0,1],[1,2],[2,0],[3,0]]) == True
assert hasCycleUndirected(4,[[0,1],[1,2],[2,3]]) == False
print('All tests passed!')

## M22. Number of Ways to Reach Destination in DAG

> 🏢 **Asked by:** Amazon, Google
Given a DAG with n nodes and directed edges, count the number of distinct paths from source to destination.

**Approach**: Topological sort + DP. dp[v] = sum of dp[u] for all predecessors u of v. Initialize dp[source]=1.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def countPathsDAG(n, edges, source, destination):
    graph=defaultdict(list); indegree=[0]*n
    rev=defaultdict(list)
    for u,v in edges: graph[u].append(v); indegree[v]+=1; rev[v].append(u)
    dp=[0]*n; dp[source]=1
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            dp[nei]+=dp[node]
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return dp[destination]

assert countPathsDAG(4,[[0,1],[0,2],[1,3],[2,3]],0,3) == 2
assert countPathsDAG(5,[[0,1],[0,2],[1,3],[2,3],[3,4]],0,4) == 2
assert countPathsDAG(3,[[0,1],[1,2]],0,2) == 1
print('All tests passed!')

## M23. Minimum Edges to Add to Make Graph Strongly Connected (Simplified)

> 🏢 **Asked by:** Amazon, Google
Given a directed graph on n nodes, find the minimum number of edges to add so that the entire graph is strongly connected.

**Approach**: Condense to SCC DAG using Kosaraju's or Tarjan's. In the condensed DAG the answer is max(nodes with in-degree 0, nodes with out-degree 0), except for trivial case.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def minEdgesForSCC(n, edges):
    if n==1: return 0
    graph=defaultdict(list); rev=defaultdict(list)
    for u,v in edges: graph[u].append(v); rev[v].append(u)
    visited=set(); order=[]
    def dfs1(node):
        visited.add(node)
        for nei in graph[node]:
            if nei not in visited: dfs1(nei)
        order.append(node)
    for i in range(n):
        if i not in visited: dfs1(i)
    comp=[0]*n; c=0; visited2=set()
    def dfs2(node, c):
        visited2.add(node); comp[node]=c
        for nei in rev[node]:
            if nei not in visited2: dfs2(nei,c)
    for node in reversed(order):
        if node not in visited2: dfs2(node,c); c+=1
    if c==1: return 0
    in_deg=[0]*c; out_deg=[0]*c
    for u,v in edges:
        if comp[u]!=comp[v]: out_deg[comp[u]]+=1; in_deg[comp[v]]+=1
    no_in=sum(1 for i in range(c) if in_deg[i]==0)
    no_out=sum(1 for i in range(c) if out_deg[i]==0)
    return max(no_in, no_out)

assert minEdgesForSCC(3,[[0,1],[1,2]]) == 1
assert minEdgesForSCC(4,[[0,1],[2,3]]) == 2
assert minEdgesForSCC(3,[[0,1],[1,2],[2,0]]) == 0
print('All tests passed!')

## M24. Find Strongly Connected Components — Kosaraju's Algorithm

> 🏢 **Asked by:** Google, Amazon
Given a directed graph, find all Strongly Connected Components (SCCs) using Kosaraju's two-pass DFS algorithm.

**Approach**: (1) DFS on original graph; push nodes to stack in finish order. (2) DFS on reversed graph in reverse finish order; each DFS tree is one SCC.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def kosarajuSCC(n, edges):
    graph=defaultdict(list); rev=defaultdict(list)
    for u,v in edges: graph[u].append(v); rev[v].append(u)
    visited=set(); stack=[]
    def dfs1(node):
        visited.add(node)
        for nei in graph[node]:
            if nei not in visited: dfs1(nei)
        stack.append(node)
    for i in range(n):
        if i not in visited: dfs1(i)
    visited2=set(); sccs=[]
    def dfs2(node, scc):
        visited2.add(node); scc.append(node)
        for nei in rev[node]:
            if nei not in visited2: dfs2(nei, scc)
    while stack:
        node=stack.pop()
        if node not in visited2:
            scc=[]; dfs2(node,scc); sccs.append(sorted(scc))
    return sccs

sccs=kosarajuSCC(5,[[0,2],[2,1],[1,0],[0,3],[3,4]])
sizes=sorted(len(s) for s in sccs)
assert sizes==[1,1,3]  # {0,1,2}, {3}, {4}
sccs2=kosarajuSCC(3,[[0,1],[1,2],[2,0]])
assert len(sccs2)==1 and len(sccs2[0])==3
print('All tests passed!')

## M25. Topological Sort With Priority — Min-Heap Queue

> 🏢 **Asked by:** Amazon, Google
Given a DAG, return topological order where among all available nodes the one with smallest index is always chosen next.

**Approach**: Standard Kahn's with a min-heap. Always extract the minimum available node.

- **Time**: O((V+E) log V)
- **Space**: O(V + E)

In [ ]:
def topoWithPriority(n, edges):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    heap=[i for i in range(n) if indegree[i]==0]
    heapq.heapify(heap); order=[]
    while heap:
        node=heapq.heappop(heap); order.append(node)
        for nei in graph[node]:
            indegree[nei]-=1
            if indegree[nei]==0: heapq.heappush(heap,nei)
    return order if len(order)==n else []

assert topoWithPriority(6,[[5,2],[5,0],[4,0],[4,1],[2,3],[3,1]]) == [4,5,0,2,3,1]
assert topoWithPriority(4,[[0,1],[0,2],[1,3],[2,3]]) == [0,1,2,3]
print('All tests passed!')

## M26. Check if Graph Has a Unique Topological Order

> 🏢 **Asked by:** Amazon, Google
A directed graph has a unique topological order iff at each step exactly one node has in-degree 0 (i.e., the topological sort is a Hamiltonian path).

**Approach**: Run Kahn's; at each step verify that exactly one node has indegree 0. If ever 0 or >1 exist simultaneously, the order is not unique.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def hasUniqueTopoOrder(n, edges):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    queue=deque(i for i in range(n) if indegree[i]==0)
    count=0
    while queue:
        if len(queue)>1: return False
        node=queue.popleft(); count+=1
        for nei in graph[node]:
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return count==n

assert hasUniqueTopoOrder(4,[[0,1],[1,2],[2,3]]) == True
assert hasUniqueTopoOrder(4,[[0,1],[0,2],[1,3],[2,3]]) == False
print('All tests passed!')

## M27. Find All Hamiltonian Paths in DAG (Small Graph)

> 🏢 **Asked by:** Amazon, Google
Given a small DAG, find all Hamiltonian paths (paths visiting every node exactly once).

**Approach**: DFS backtracking respecting directed edges; record path when all nodes are visited.

- **Time**: O(V! worst case)
- **Space**: O(V)

In [ ]:
def hamiltonianPathsDAG(n, edges):
    graph=defaultdict(list)
    for u,v in edges: graph[u].append(v)
    results=[]
    def dfs(node, path, visited):
        if len(path)==n: results.append(list(path)); return
        for nei in graph[node]:
            if nei not in visited:
                visited.add(nei); path.append(nei)
                dfs(nei, path, visited)
                path.pop(); visited.discard(nei)
    for start in range(n):
        dfs(start,[start],{start})
    return results

paths=hamiltonianPathsDAG(4,[[0,1],[0,2],[1,3],[2,3]])
assert [0,1,3] not in paths  # can't reach 2
# path 0->1->3 only visits 3 nodes
assert len(paths)==0  # no HP since 0->1->3 skips 2 and 0->2->3 skips 1
paths2=hamiltonianPathsDAG(3,[[0,1],[1,2],[0,2]])
assert [0,1,2] in paths2
print('All tests passed!')

## M28. Minimum Cost to Finish All Courses in Order

> 🏢 **Asked by:** Amazon, Google
Given n courses each with a cost and prerequisite chains, find the minimum total cost to complete all courses (taking prerequisites in order).

**Approach**: Topological sort + DP accumulation. Process nodes in topo order; accumulate cost along the critical path.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def minCostAllCourses(n, costs, prerequisites):
    graph=defaultdict(list); indegree=[0]*n
    for task,pre in prerequisites: graph[pre].append(task); indegree[task]+=1
    dp=list(costs)  # dp[i] = min cost to complete course i including all its prereqs
    queue=deque(i for i in range(n) if indegree[i]==0)
    total_processed=0
    while queue:
        node=queue.popleft(); total_processed+=1
        for nei in graph[node]:
            dp[nei]=max(dp[nei], dp[node]+costs[nei])
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    if total_processed!=n: return -1
    return sum(costs)  # all courses must be taken; total cost is simply sum

assert minCostAllCourses(3,[1,2,3],[[1,0],[2,1]]) == 6
assert minCostAllCourses(3,[5,3,4],[]) == 12
print('All tests passed!')

## M29. Count Paths From Source to Destination in DAG

> 🏢 **Asked by:** Amazon, Google
Given a DAG and source/destination nodes, count the number of distinct paths from source to destination.

**Approach**: Memoized DFS. From each node, the count of paths to destination is the sum of counts from all its neighbors.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def countPathsToDestination(n, edges, source, destination):
    graph=defaultdict(list)
    for u,v in edges: graph[u].append(v)
    memo={}
    def dfs(node):
        if node==destination: return 1
        if node in memo: return memo[node]
        total=sum(dfs(nei) for nei in graph[node])
        memo[node]=total
        return total
    return dfs(source)

assert countPathsToDestination(4,[[0,1],[0,2],[1,3],[2,3]],0,3) == 2
assert countPathsToDestination(5,[[0,1],[0,2],[1,3],[2,3],[3,4]],0,4) == 2
assert countPathsToDestination(3,[[0,1],[1,2]],0,2) == 1
print('All tests passed!')

## M30. Course Schedule IV (LC 1462) — Reachability Queries

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given n courses and prerequisite pairs, answer q queries: for each [u,v] determine if u is a prerequisite of v.

**Approach**: Topological sort + transitive closure. Take the topological order, then walk it **backwards**: by the time a node is reached its successors are final, so `reachable[u]` is the union of its neighbours and their reachable sets. Walking forwards instead reads sets that are still being built.

- **Time**: O(V·E) to propagate, O(V²) space for the sets
- **Space**: O(V²)

The Floyd–Warshall closure below is the shorter alternative: O(V³), fine while V is small, and useful as a cross-check on the topological version.

In [ ]:
def checkIfPrerequisite(numCourses, prerequisites, queries):
    # Reachability by propagation in *reverse* topological order: a node's
    # successors are finished before the node itself, so unioning their sets is
    # enough. Doing it in forward order -- the tempting version -- reads sets
    # that are still being filled in and silently under-reports.
    graph = defaultdict(list)
    indegree = [0] * numCourses
    for u, v in prerequisites:
        graph[u].append(v)
        indegree[v] += 1

    order, queue = [], deque(i for i in range(numCourses) if indegree[i] == 0)
    while queue:
        node = queue.popleft()
        order.append(node)
        for nei in graph[node]:
            indegree[nei] -= 1
            if indegree[nei] == 0:
                queue.append(nei)

    reachable = [set() for _ in range(numCourses)]
    for node in reversed(order):
        for nei in graph[node]:
            reachable[node].add(nei)
            reachable[node] |= reachable[nei]
    return [v in reachable[u] for u, v in queries]

def checkIfPrerequisiteFloyd(numCourses, prerequisites, queries):
    # Floyd-Warshall transitive closure: shorter to write, O(V^3) instead of
    # O(V*E), and worth knowing when V is small.
    reach = [[False] * numCourses for _ in range(numCourses)]
    for u, v in prerequisites:
        reach[u][v] = True
    for k in range(numCourses):
        for i in range(numCourses):
            for j in range(numCourses):
                if reach[i][k] and reach[k][j]:
                    reach[i][j] = True
    return [reach[u][v] for u, v in queries]

for solve in (checkIfPrerequisite, checkIfPrerequisiteFloyd):
    assert solve(2, [[1,0]], [[0,1],[1,0]]) == [False, True]
    assert solve(3, [[1,2],[1,0],[2,0]], [[1,0],[1,2]]) == [True, True]
    assert solve(2, [[1,0]], [[1,1]]) == [False]           # a course is not its own prereq
    assert solve(5, [[0,1],[1,2],[2,3],[3,4]], [[0,4],[4,0]]) == [True, False]
    assert solve(3, [], [[0,1]]) == [False]                # no prerequisites at all
print("All tests passed!")

## Hard Problems (11-20)

## H11. Longest Path in DAG With Node Weights

> 🏢 **Asked by:** Amazon, Google
Given a DAG with n nodes, directed edges, and node weights, find the length of the longest path (sum of node weights along the path).

**Approach**: Topological sort + DP. dp[v] = max weight path ending at v = weights[v] + max(dp[u]) for all predecessors u.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def longestPathDAG(n, edges, weights):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    dp=list(weights)
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            dp[nei]=max(dp[nei], dp[node]+weights[nei])
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return max(dp)

# 0(1)->1(3)->2(2): best path = 1+3+2=6
assert longestPathDAG(3,[[0,1],[1,2]],[1,3,2]) == 6
# 0(5)->2(1) or 1(3)->2(1): best=5+1=6
assert longestPathDAG(3,[[0,2],[1,2]],[5,3,1]) == 6
print('All tests passed!')

## H12. Kosaraju's Algorithm for Strongly Connected Components

> 🏢 **Asked by:** Google, Amazon
Implement Kosaraju's full algorithm to find all SCCs and return them as lists of nodes.

**Approach**: Two-pass DFS. Pass 1 on original graph builds finish-order stack. Pass 2 on reversed graph processes nodes in reverse finish order — each DFS tree is an SCC.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def kosarajuFull(n, edges):
    graph=defaultdict(list); rev=defaultdict(list)
    for u,v in edges: graph[u].append(v); rev[v].append(u)
    visited=set(); finish_stack=[]
    def dfs1(u):
        st=[(u,False)]
        while st:
            node,processed=st.pop()
            if processed: finish_stack.append(node); continue
            if node in visited: continue
            visited.add(node); st.append((node,True))
            for nei in graph[node]: st.append((nei,False))
    for i in range(n):
        if i not in visited: dfs1(i)
    visited2=set(); sccs=[]
    def dfs2(u):
        scc=[]; st=[u]
        while st:
            node=st.pop()
            if node in visited2: continue
            visited2.add(node); scc.append(node)
            for nei in rev[node]: st.append(nei)
        return scc
    while finish_stack:
        node=finish_stack.pop()
        if node not in visited2:
            sccs.append(sorted(dfs2(node)))
    return sccs

sccs=kosarajuFull(5,[[0,2],[2,1],[1,0],[0,3],[3,4]])
sizes=sorted(len(s) for s in sccs)
assert sizes==[1,1,3]
sccs2=kosarajuFull(4,[[0,1],[1,2],[2,3],[3,0]])
assert len(sccs2)==1 and len(sccs2[0])==4
print('All tests passed!')

## H13. Minimum Vertices to Traverse All Nodes (DAG Covering)

> 🏢 **Asked by:** Amazon, Google
Given a DAG, find the minimum number of source vertices needed such that every node is reachable from at least one source (minimum path cover equivalent for DAG).

**Approach**: Nodes with in-degree 0 must be starting points. Any node with in-degree > 0 can be reached from its predecessors, so we only need in-degree-0 nodes.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def minVerticesToTraverseAll(n, edges):
    has_predecessor=set()
    for u,v in edges: has_predecessor.add(v)
    return [i for i in range(n) if i not in has_predecessor]

assert sorted(minVerticesToTraverseAll(6,[[0,1],[0,2],[2,5],[3,4],[4,2]])) == [0,3]
assert sorted(minVerticesToTraverseAll(4,[[0,1],[1,2],[2,3]])) == [0]
assert sorted(minVerticesToTraverseAll(3,[])) == [0,1,2]
print('All tests passed!')

## H14. Topological Sort for Matrix Chain Multiplication Dependencies

> 🏢 **Asked by:** Amazon, Google
Given a sequence of matrix multiplication operations with dependencies (result of one feeds into another), find the optimal evaluation order using topological sort.

**Approach**: Model each matrix multiplication as a node; dependencies as edges. Return topological order — this is the valid evaluation order.

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def matrixChainOrder(n, dependencies):
    # dependencies[i] = list of operations that must complete before operation i
    graph=defaultdict(list); indegree=[0]*n
    for v, prereqs in enumerate(dependencies):
        for u in prereqs: graph[u].append(v); indegree[v]+=1
    queue=deque(i for i in range(n) if indegree[i]==0)
    order=[]
    while queue:
        node=queue.popleft(); order.append(node)
        for nei in graph[node]:
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return order if len(order)==n else []

# Op0 and Op1 are independent; Op2 needs Op0; Op3 needs Op1 and Op2
deps=[[],[],[0],[1,2]]
order=matrixChainOrder(4,deps)
assert len(order)==4
assert order.index(0)<order.index(2)
assert order.index(2)<order.index(3)
assert order.index(1)<order.index(3)
print('All tests passed!')

## H15. Critical Path Method (Find Critical Path Length in Project DAG)

> 🏢 **Asked by:** Amazon, Google
Given a project DAG with task durations and dependencies, find the length of the critical path (minimum project duration).

**Approach**: Topological sort + forward pass (earliest start time). earliest_finish[v] = duration[v] + max(earliest_finish[u]) for all predecessors u. Critical path = max(earliest_finish).

- **Time**: O(V + E)
- **Space**: O(V)

In [ ]:
def criticalPathLength(n, durations, dependencies):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in dependencies: graph[u].append(v); indegree[v]+=1
    earliest=[0]*n
    for i in range(n): earliest[i]=durations[i]
    queue=deque(i for i in range(n) if indegree[i]==0)
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            earliest[nei]=max(earliest[nei], earliest[node]+durations[nei])
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return max(earliest)

# A(3)->C(2)->D(1), B(4)->C(2)->D(1). Critical path = B+C+D = 4+2+1=7
assert criticalPathLength(4,[3,4,2,1],[[0,2],[1,2],[2,3]]) == 7
assert criticalPathLength(3,[1,2,3],[[0,2],[1,2]]) == 5
print('All tests passed!')

## H16. Job Sequencing With Deadlines — Greedy + Topological Insight

> 🏢 **Asked by:** Amazon, Google
Given jobs with profit and deadline (must be completed by deadline), find maximum profit schedule assuming each job takes 1 unit time.

**Approach**: Sort jobs by profit descending. Greedily assign each job to the latest available slot <= its deadline.

- **Time**: O(n²)
- **Space**: O(n)

In [ ]:
def jobSequencing(jobs):
    # jobs = [(profit, deadline), ...]
    jobs_sorted=sorted(jobs, key=lambda x:-x[0])
    max_deadline=max(d for _,d in jobs)
    slots=[False]*(max_deadline+1)
    total_profit=count=0
    for profit,deadline in jobs_sorted:
        for slot in range(min(deadline,max_deadline),0,-1):
            if not slots[slot]:
                slots[slot]=True
                total_profit+=profit; count+=1
                break
    return count, total_profit

count,profit=jobSequencing([(100,2),(19,1),(27,2),(25,1),(15,3)])
assert count==3 and profit==142
count2,profit2=jobSequencing([(20,1),(15,2),(10,1),(5,3),(1,3)])
assert profit2==40
print('All tests passed!')

## H17. Shortest Path in DAG (Single Source)

> 🏢 **Asked by:** Amazon, Google
Given a weighted DAG with n nodes and edges [u,v,w], find the shortest distances from a source node to all other nodes.

**Approach**: Topological sort first; then process nodes in topo order relaxing edges (no cycles so no need for Dijkstra's priority queue).

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def shortestPathDAG(n, weighted_edges, source):
    graph=defaultdict(list); indegree=[0]*n
    for u,v,w in weighted_edges: graph[u].append((v,w)); indegree[v]+=1
    queue=deque(i for i in range(n) if indegree[i]==0)
    topo_order=[]
    ind=list(indegree)
    while queue:
        node=queue.popleft(); topo_order.append(node)
        for nei,w in graph[node]:
            ind[nei]-=1
            if ind[nei]==0: queue.append(nei)
    dist=[float('inf')]*n; dist[source]=0
    for node in topo_order:
        if dist[node]==float('inf'): continue
        for nei,w in graph[node]:
            if dist[node]+w < dist[nei]: dist[nei]=dist[node]+w
    return dist

d=shortestPathDAG(6,[[0,1,5],[0,2,3],[1,3,6],[1,2,2],[2,4,4],[2,5,2],[2,3,7],[3,5,1],[4,5,4]],0)
assert d[0]==0 and d[5]==5
print('All tests passed!')

## H18. All Topological Sorts — Enumerate All Valid Orderings

> 🏢 **Asked by:** Amazon, Google
Given a DAG, enumerate all valid topological orderings. Return them all sorted lexicographically.

**Approach**: Backtracking with indegree tracking. At each step try every 0-indegree node; recurse; undo choice.

- **Time**: O(V! worst case)
- **Space**: O(V)

In [ ]:
def allTopologicalSorts(n, edges):
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    results=[]
    def backtrack(path, ind, visited):
        if len(path)==n: results.append(list(path)); return
        for node in range(n):
            if ind[node]==0 and node not in visited:
                visited.add(node); path.append(node)
                for nei in graph[node]: ind[nei]-=1
                backtrack(path, ind, visited)
                for nei in graph[node]: ind[nei]+=1
                path.pop(); visited.discard(node)
    backtrack([], indegree[:], set())
    return sorted(results)

sorts=allTopologicalSorts(4,[[0,1],[0,2],[1,3],[2,3]])
assert [0,1,2,3] in sorts and [0,2,1,3] in sorts
assert len(sorts)==2
print('All tests passed!')

## H19. Check if Directed Graph is a Tree

> 🏢 **Asked by:** Amazon, Google
A directed graph is a tree if: (1) it has exactly n-1 edges, (2) it is connected (as undirected), and (3) it has exactly one node with in-degree 0 (the root).

**Approach**: Check edge count, check exactly one root (indegree 0), then verify all nodes are reachable from root via BFS.

- **Time**: O(V + E)
- **Space**: O(V + E)

In [ ]:
def isDirectedTree(n, edges):
    if len(edges)!=n-1: return False
    graph=defaultdict(list); indegree=[0]*n
    for u,v in edges: graph[u].append(v); indegree[v]+=1
    roots=[i for i in range(n) if indegree[i]==0]
    if len(roots)!=1: return False
    root=roots[0]
    visited={root}; queue=deque([root])
    while queue:
        node=queue.popleft()
        for nei in graph[node]:
            if nei not in visited: visited.add(nei); queue.append(nei)
    return len(visited)==n

assert isDirectedTree(4,[[0,1],[0,2],[1,3]]) == True
assert isDirectedTree(4,[[0,1],[1,2],[2,3],[3,0]]) == False  # cycle
assert isDirectedTree(4,[[0,1],[0,2]]) == False  # only 2 edges for 4 nodes
assert isDirectedTree(4,[[0,1],[0,2],[1,3],[2,3]]) == False  # node 3 has 2 parents
print('All tests passed!')

## H20. Build a Complete Ranking From Partial Rankings (LC 444 Extended)

> 🏢 **Asked by:** Amazon, Google
Given a sequence of numbers (1..n) and a subsequence that must appear in this order, determine if a valid complete permutation exists and return one if so.

**Approach**: The subsequence imposes ordering constraints (edges in a DAG). Use topological sort; if a valid ordering exists (no cycle) and produces length n, return it.

- **Time**: O(n + m) where m = len(subsequence)
- **Space**: O(n)

In [ ]:
def sequenceReconstruction(nums, sequences):
    n=len(nums)
    graph=defaultdict(set); indegree={i:0 for i in range(1,n+1)}
    for seq in sequences:
        for i in range(len(seq)-1):
            u,v=seq[i],seq[i+1]
            if v not in graph[u]:
                graph[u].add(v); indegree[v]+=1
    queue=deque(i for i in range(1,n+1) if indegree[i]==0)
    order=[]
    while queue:
        if len(queue)>1: return False  # multiple choices = not unique
        node=queue.popleft(); order.append(node)
        for nei in graph[node]:
            indegree[nei]-=1
            if indegree[nei]==0: queue.append(nei)
    return order==nums

assert sequenceReconstruction([1,2,3],[[1,2],[1,3],[2,3]]) == True
assert sequenceReconstruction([1,2,3],[[1,2]]) == False
assert sequenceReconstruction([4,1,5,2,6,3],[[5,2,6,3],[4,1,5,2]]) == True
print('All tests passed!')